# 05 · Optimisation Walkthrough

The end-to-end pipeline: train the ensemble, project the next 5 gameweeks, solve the multi-gameweek MILP. Inspect the solver status, the recommended XI, the captain choice, and the objective's implicit trade between free transfers and point hits.

In [ ]:
from gaffer.providers.historical_csv import HistoricalCsvProvider
from gaffer.providers.fpl_api import LiveFplApiProvider
from gaffer.services.model_cache import train_or_load_ensembles
from gaffer.services.prediction_service import predict_projections
from gaffer.services.optimization_service import optimize_squad

fpl = LiveFplApiProvider()
hist = HistoricalCsvProvider()
point, quantile = train_or_load_ensembles(fpl, hist)
proj = predict_projections(fpl=fpl, historical=hist, point_model=point, quantile_model=quantile, horizon_gws=5)
proj.projections.head()

In [ ]:
result = optimize_squad(
    projections=proj.projections,
    players=proj.players,
    start_gw=fpl.get_current_gw(),
    horizon=5,
)
print(f'Status: {result.solver_status} · net points: {result.total_expected_points:.1f}')
for plan in result.plans:
    print(f'GW{plan.gameweek}: XI={len(plan.starting_xi)}, captain={plan.captain_id}, transfers={len(plan.transfers_in)}')

In [ ]:
# Inspect the first-GW XI in detail.
plan = result.plans[0]
xi = proj.players.loc[list(plan.starting_xi)].copy()
xi['xPts'] = proj.projections.xs(plan.gameweek, level='gameweek').loc[xi.index, 'expected_points']
xi.sort_values('xPts', ascending=False)

That's the full pipeline. The Streamlit app (`streamlit run app/Home.py`) wraps these calls in an interactive UI.